# Roche LightCycler 480 II

The Roche **LightCycler 480 II** is a 96- or 384-well qPCR instrument: a single-zone thermal block,
and a camera that reads every well's fluorescence at the steps you ask it to.

| | |
|---|---|
| Plates | 96 or 384 wells |
| Reads | every well at once, at any step |
| Interface | Ethernet, TCP ports 5100-5105 |
| Driver | `pylabrobot.roche.lightcycler.LightCycler480` |

```{warning}
This driver is a Python port of a Go driver that ran experiments on a LightCycler 480 II with
firmware 1.8.6.1501. The port itself has not yet run on an instrument. Only the SYBR Green / FAM
filter set has been seen on the wire. If it works on yours, please say so on
[discuss.pylabrobot.org](https://discuss.pylabrobot.org) or in a pull request.
```

```{device-card} roche-lightcycler-480-ii
```

## How it talks

The instrument listens on five TCP ports from 5100, each message a 2-byte type, a 2-byte length and
newline-separated ASCII fields. The driver keeps the connection alive, acknowledges the events the
instrument sends, defines each experiment as programs of steps, and collects each fluorescence read
as it is taken. The instrument's tray is moved by the button on its front; nothing on the wire
moves it, so to move it from software, mount a [SwitchBot Bot](https://www.switch-bot.com/) on the
button and give the driver a `SwitchBotBot` (needs `bleak`).

## Physical setup

Connect the instrument's Ethernet port to your computer. Its address is static, 192.168.95.41, so
give your computer an address on the same subnet, 192.168.95.42 say.

## Setup

`simulation=True` drives a simulated instrument instead, with a simulated tray button.

In [ ]:
from pylabrobot.roche.lightcycler import LightCycler480, SwitchBotBot

lc = LightCycler480(host="192.168.95.41", tray_button=SwitchBotBot("C1:23:45:67:89:AB"))
# lc = LightCycler480(simulation=True)
await lc.setup()
await lc.request_status()

## Loading a plate

Bring the tray out. Each tray call does nothing when the tray is already where it is asked to be.

In [ ]:
await lc.open_tray()

Put the plate on the tray, then close it.

In [ ]:
await lc.close_tray()
await lc.request_tray_state()

## Running an experiment

An experiment is a thermocycling `Protocol`. A `LightCyclerStep` with `acquire=True` reads every
well once the block is at its temperature; a plain `Step` only holds. The steps of a stage hold for
one time, as the instrument holds each program's steps.

In [ ]:
from pylabrobot.legacy.thermocycling.standard import Protocol, Stage
from pylabrobot.roche.lightcycler import LightCyclerStep

protocol = Protocol([
  Stage([LightCyclerStep(temperature=[95], hold_seconds=30)], repeats=1),
  Stage([
    LightCyclerStep(temperature=[95], hold_seconds=10),
    LightCyclerStep(temperature=[60], hold_seconds=10, acquire=True),
  ], repeats=40),
])
experiment = await lc.run_experiment(protocol, well_count=384, volume=20)
experiment.status, len(experiment.acquisitions)

Each read has the block's temperature and every well's raw fluorescence.

In [ ]:
first = experiment.acquisitions[0]
first.temperature, first.values[:8]

## Following an experiment

`start_experiment` returns the experiment's id at once and leaves it running;
`watch_experiment`, `wait_for_experiment` and `get_experiment` follow it by that id.

In [ ]:
experiment_id = await lc.start_experiment(protocol)
async for progress in lc.watch_experiment(experiment_id):
  print(progress.status, len(progress.acquisitions), "of", progress.acquisitions_total)

## Serving it

`python -m pylabrobot.roche.lightcycler.server --device 192.168.95.41 --switchbot C1:23:45:67:89:AB`
serves the instrument over Connect, on port 8097; `--simulation` serves a simulated one.
`LightCycler480(lightcycler_server="http://host:8097")` then drives it with the same methods. An
experiment runs on the server and is rejoined by its id after a dropped connection.

## Stopping

In [ ]:
await lc.stop()